# 06 · Vendas manuais da Batel: raio-x

Fonte: `batel_vendas_manuais.csv` (a planilha `vendas_batel_MANUAL_marco.xlsx`, gerada por `padronizar.py`).

Contexto (`REGRAS_OPERACAO.txt`): a Batel ficou **sem PDV de 02/03 a 22/03/2026**. As vendas foram anotadas nessa planilha, e a gerente relançou no sistema "as principais" quando ele voltou.

**Como é o Excel original** (só olhar; nunca salvar pelo Excel): título "lançar tudo quando voltar!!" em cima; a data foi digitada de vários jeitos ("02/03", "03/03/2026") e **o Excel transformou parte delas em data trocando dia e mês** ("05/03" virou 3 de maio); as peças estão em texto livre; no fim há subtotais e o total; a aba "Plan2" diz "conferir com extrato da maquininha", "semana 2 faltou 1 venda pix?".

> Antes de commitar: *Kernel → Restart & Clear Output* (as tabelas têm CPF, e-mail e telefone).

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

In [ ]:
batel = pd.read_csv(STAGING / "batel_vendas_manuais.csv", dtype=str)
vendas = pd.read_csv(STAGING / "banco_vendas.csv", dtype=str)
batel["valor"] = pd.to_numeric(batel["VALOR"])
batel.head()

## 1. O que é cada linha?

Uma venda anotada à mão. Exceto 4 linhas sem data: são subtotais e o total.

In [ ]:
sem_data = batel[batel["DATA"].isna()]
print("linhas:", len(batel), "| vendas:", batel["DATA"].notna().sum(), "| subtotais/total:", len(sem_data))
sem_data[["linha_origem", "VALOR"]]

## 2 e 3. Colunas e problemas

In [ ]:
vendas_batel = batel[batel["DATA"].notna()]
# formatos da data (9 = dígito); "data" = célula que o Excel converteu (com dia e mês trocados)
formato = vendas_batel["DATA"].map(lambda s: "convertida pelo Excel" if s.endswith("00:00:00") else pd.Series([s]).str.replace(r"\d", "9", regex=True)[0])
display(formato.value_counts())
print("datas convertidas:", vendas_batel.loc[formato == "convertida pelo Excel", "DATA"].value_counts().to_dict())
print("OBS:", vendas_batel["OBS"].value_counts().to_dict(), "| PGTO:", vendas_batel["PGTO"].value_counts().to_dict())
print("sem CPF:", vendas_batel["CPF"].isna().sum(), "| soma das vendas: R$", round(vendas_batel["valor"].sum(), 2))
vendas_batel[["PEÇAS", "QTD", "VALOR"]].head()

## 4. Como se liga ao sistema?

In [ ]:
vendas["dia"] = pd.to_datetime(vendas["criado_em"]).dt.tz_convert("America/Sao_Paulo")
batel_sistema = vendas[(vendas["loja_id"] == "4") & (vendas["dia"].dt.strftime("%Y-%m") == "2026-03")]
print("sistema, Batel em março:", len(batel_sistema), "vendas, R$", round(pd.to_numeric(batel_sistema["total"]).sum(), 2))
display(batel_sistema.groupby(batel_sistema["dia"].dt.strftime("%d/%m")).size())

# relançamentos: procurar na planilha pelo CPF (tirando os vazios antes de cruzar)
relancadas = vendas[(vendas["loja_id"] == "4") & vendas["obs"].fillna("").str.contains("retro")]
achadas = relancadas.dropna(subset=["cliente_valor"]).merge(vendas_batel.dropna(subset=["CPF"]), left_on="cliente_valor", right_on="CPF")
print("relançamentos:", len(relancadas), "| achados na planilha pelo CPF:", achadas["id"].nunique())
achadas[["id", "criado_em", "total", "DATA", "VALOR"]]

## Desafios da revisão

In [ ]:
# os subtotais batem com as vendas de cada bloco?
batel["linha"] = batel["linha_origem"].str.extract(r"!(\d+)")[0].astype(int)
inicio = batel["linha"].min()
for _, subtotal in batel[batel["DATA"].isna()].sort_values("linha").iterrows():
    bloco = batel[(batel["linha"] >= inicio) & (batel["linha"] < subtotal["linha"]) & batel["DATA"].notna()]
    print(f"até a linha {subtotal['linha']}: {len(bloco)} vendas somam {bloco['valor'].sum():.2f} | a planilha diz {subtotal['valor']:.2f}")
    inicio = subtotal["linha"]
sem_canceladas = vendas_batel[~vendas_batel["OBS"].fillna("").str.contains("cancel|estorn")]
print("todas as vendas:", round(vendas_batel["valor"].sum(), 2), "| sem a cancelada e as estornadas:", round(sem_canceladas["valor"].sum(), 2))

## Anotações para o dicionário de dados

**O que é cada linha**
- Uma venda anotada à mão (72), mais 4 linhas sem data que são subtotais e total (`MARÇO!31` 10.397,83; `MARÇO!56` 9.755,02; `MARÇO!78` 7.063,44; `MARÇO!80` 27.216,29). **Não são vendas.**

**Colunas**

| Coluna | O que é | Problemas vistos |
|---|---|---|
| `DATA` | dia da venda | 3 formatos: `03/03/2026` (26), `20/03` sem ano (20) e **26 que o Excel converteu trocando dia e mês** (`2026-05-03` = 05/03; todas têm dia 03, porque são de março) |
| `HORA` | hora | 14 vazias |
| `CLIENTE` / `CPF` | quem comprou | 30 sem nome, 32 sem CPF |
| `PEÇAS` | o que foi vendido, em texto livre | **sem SKU** ("2x pochete nylon branco U + camisa linho leve lilás G"): não dá para baixar o estoque com certeza |
| `QTD` / `VALOR` | quantidade e valor | soma das vendas: R$ 28.025,99 |
| `PGTO` | forma de pagamento | `credito 2x`, `débito`, `dinheiro`, `crédito`, `pix` |
| `OBS` | recado | `lancei no vulto+` (13), `desconto gerente` (6), `estornado na maquininha` (2), `cancelou` (1) |

**Ligação com o sistema**
- O sistema tem, da Batel em março, 51 vendas (R$ 16.723,15): 1 em 01/03, 1 em 13/03 (com o PDV fora do ar?) e as demais depois de 23/03, quando ele voltou.
- 16 relançamentos (`lancamento retroativo`, de 23/03 a 02/04) com a **data do relançamento**, não da venda: parte cai em abril. 9 acham a venda da planilha pelo CPF, com valores às vezes diferentes (310,00 × 309,80; 329,90 × 296,91).
- **Risco:** somar a planilha e os relançamentos conta a mesma venda duas vezes; usar só o sistema perde as vendas não relançadas.

**Problemas da Batel (lista única; as decisões estão no registro de decisões)**

| # | Problema | Quantos | Afeta |
|---|---|---|---|
| 1 | Subtotais e total no meio das vendas | 4 | faturamento (se somados, quase triplicam) |
| 2 | Datas com dia e mês trocados pelo Excel; datas sem ano | 26 + 20 | dia da venda |
| 3 | Relançamentos com a data do relançamento (parte em abril) | 16 | faturamento de março × abril |
| 4 | Planilha × relançamentos: duplicar ou perder vendas | 72 × 16 | faturamento da Batel |
| 5 | Peças sem SKU | 72 | baixa de estoque |
| 6 | Cancelada e estornadas anotadas só na OBS | 3 | faturamento |
| 7 | "Semana 2 faltou 1 venda pix?" (aba Plan2) | ? | faturamento |

**Desafios da revisão**
- **Subtotais:** semana 1 = R$ 10.397,83 (as vendas somam R$ 11.207,53); semanas 2 e 3 batem (R$ 9.755,02 e R$ 7.063,44). **O total da planilha (R$ 27.216,29) é a soma das vendas sem a cancelada e as 2 estornadas** (R$ 28.025,99 − R$ 809,70): a gerente já tirou essas três do total.
- **"Semana 2 faltou 1 venda pix?":** não aparece nos números (a semana 2 bate com o subtotal). Se a venda existiu, não foi anotada em lugar nenhum: não dá para recuperar, só citar como margem de erro.

*(continuar preenchendo ao longo da análise)*